In [51]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from importlib import reload

# 重新加载所有模块
import loaders, filters, windowers, splitters, normalizers,constants
reload(constants)
reload(loaders)
reload(filters)
reload(windowers)
reload(splitters)
reload(normalizers)

from loaders import CSVLoader
from filters import DataFilter
from windowers import WindowGenerator
from splitters import DataSplitter
from normalizers import UnifiedScaler

loader = CSVLoader("../../data for sharing_csv", ["P2","P3","P4","P5","P6","P7","P8"])

# Checkpoint 1: 验证发现 7 个地点
len(loader.location_files) #"应发现 7 个地点 CSV"


7

In [41]:

# Checkpoint 3: 合并所有地点
df_merged = loader.load_and_merge()
len(df_merged) 
df_merged["location"].nunique()

7

In [46]:
# Checkpoint 4: 过滤异常数据

# 使用之前加载的 df_merged 真实数据
print("原始数据:")
print(f"形状：{df_merged.shape}")

filter_obj = DataFilter()
df_filtered = filter_obj.filter_outliers(df_merged)

print("\n过滤后数据:")
print(f"形状：{df_filtered.shape}")
# print(df_filtered.iloc[0])
# Checkpoint 5: 标签映射
y = filter_obj.map_labels(df_filtered)

原始数据:
形状：(86830, 18)

过滤后数据:
形状：(85393, 18)


In [48]:
# Checkpoint 6: 测试 WindowGenerator

# 使用过滤后的真实数据
print("输入数据形状:", df_filtered.shape)

window_gen = WindowGenerator(window_size=10, max_satellites=20)

# 生成 STCA 双通道输入
X_temporal, X_spatial, y, locations = window_gen.generate_stca_inputs(df_filtered)

print("时间通道输入形状:", X_temporal.shape)  # (N, 10, 4)
print("空间通道输入形状:", X_spatial.shape)  # (N, 20, 4)
print("标签形状:", y.shape) # 最后时刻标签
print("地点数组形状:", locations.shape) # 最后时刻地点
print()
print("标签分布：NLOS =", np.sum(y==0), ", LOS =", np.sum(y==1))


输入数据形状: (85393, 18)
时间通道输入形状: (84351, 10, 4)
空间通道输入形状: (84351, 20, 4)
标签形状: (84351,)
地点数组形状: (84351,)

标签分布：NLOS = 42161 , LOS = 42190


In [49]:
# 步骤 4: 数据划分 (Outdomain)
print("\n[步骤 4] Outdomain 数据划分...")
splitter = DataSplitter(random_seed=42)
result = splitter.split_outdomain(X_temporal, y, locations, X_spatial)
X_train_t, X_val_t, X_test_t, X_train_s, X_val_s, X_test_s, y_train, y_val, y_test = result
print(f"  Train: temporal={X_train_t.shape}, spatial={X_train_s.shape}")
print(f"  Val:   temporal={X_val_t.shape}, spatial={X_val_s.shape}")
print(f"  Test:  temporal={X_test_t.shape}, spatial={X_test_s.shape}")


[步骤 4] Outdomain 数据划分...
  Train: temporal=(45602, 10, 4), spatial=(45602, 20, 4)
  Val:   temporal=(13010, 10, 4), spatial=(13010, 20, 4)
  Test:  temporal=(25739, 10, 4), spatial=(25739, 20, 4)


In [54]:
# 步骤 5: 标准化
print("\n[步骤 5] 数据标准化 (仅使用训练集拟合)...")
scaler = UnifiedScaler.from_data(X_train_t, X_train_s)

# 时间特征标准化
X_train_t_scaled = scaler.transform(X_train_t)
X_val_t_scaled = scaler.transform(X_val_t)
X_test_t_scaled = scaler.transform(X_test_t)

# 空间特征也需要标准化！
X_train_s_scaled = scaler.transform(X_train_s)
X_val_s_scaled = scaler.transform(X_val_s)
X_test_s_scaled = scaler.transform(X_test_s)

print(f"  标准化后训练集均值 (temporal): {X_train_t_scaled.mean():.6f}")
print(f"  标准化后训练集均值 (spatial):  {X_train_s_scaled.mean():.6f}")


[步骤 5] 数据标准化 (仅使用训练集拟合)...
  标准化后训练集均值 (temporal): 0.359184
  标准化后训练集均值 (spatial):  -0.179592


In [56]:
# 步骤 6: 创建 DataLoader
print("\n[步骤 6] 创建 PyTorch DataLoader...")
batch_size = 32
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_t_scaled),   # 标准化后的时间特征
    torch.FloatTensor(X_train_s_scaled),   # 标准化后的空间特征
    torch.LongTensor(y_train)
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f"  Dataset 大小：{len(train_dataset)}")


[步骤 6] 创建 PyTorch DataLoader...
  Dataset 大小：45602


In [53]:
# 步骤 7: 验证一个 batch
print("\n[步骤 7] 验证模型输入...")
for X_t_batch, X_s_batch, y_batch in train_loader:
    print(f"  X_t_batch: {X_t_batch.shape}")
    print(f"  X_s_batch: {X_s_batch.shape}")
    print(f"  y_batch:   {y_batch.shape}")
    break

print("\n" + "=" * 70)
print("[OK] 完整数据流水线测试通过！")
print("=" * 70)


[步骤 7] 验证模型输入...
  X_t_batch: torch.Size([32, 10, 4])
  X_s_batch: torch.Size([32, 20, 4])
  y_batch:   torch.Size([32])

[OK] 完整数据流水线测试通过！
